# OLX Cars Content-Based Recommendation

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [3]:
df = pd.read_csv("OLX_cars_dataset00.csv", low_memory=False)

In [4]:
df["Car Features"] = df["Car Features"].fillna("")
df["Description"] = df["Description"].fillna("")
df["Make"] = df["Make"].fillna("")
df["Model"] = df["Model"].fillna("")

In [5]:
def create_car_soup(x):
    return (
        str(x["Make"])
        + " "
        + str(x["Model"])
        + " "
        + str(x["Car Name"])
        + " "
        + str(x["Fuel"])
        + " "
        + str(x["Transmission"])
        + " "
        + str(x["Car Features"])
    )

In [6]:
df["soup"] = df.apply(create_car_soup, axis=1)

In [7]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["soup"])

In [8]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [9]:
indices = pd.Series(df.index, index=df["Car Name"]).drop_duplicates()

In [10]:
def get_car_recommendations(car_name, cosine_sim=cosine_sim):
    if car_name not in indices:
        return f"'{car_name}' veri setinde bulunamadı."

    idx = indices[car_name]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]
    car_indices = [i[0] for i in sim_scores]

    return df[["Car Name", "Make", "Model", "Price", "Year"]].iloc[car_indices]

In [11]:
sample_car = df["Car Name"].iloc[0]
print(f"\nTarget Car: {sample_car}")
print("\nRecommended Similar Cars:")
print(get_car_recommendations(sample_car))


Target Car: fresh import Passo 2021model

Recommended Similar Cars:
                Car Name    Make  Model    Price  Year
2500               passo  Toyota  Passo  3000000  2015
58          Toyota Passo  Toyota  Passo  3750000  2018
2172        Toyota Passo  Toyota  Passo  4350000  2021
6707  Passo 1.3 for sale  Toyota  Passo  1810000  2005
7429   Toyota Passo 2012  Toyota  Passo  2520000  2012
